In [ ]:
# pip install -r requirements.txt

In [ ]:
import os
import torch
import torch.nn as nn
import torchaudio
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset

In [ ]:
import gdown
import zipfile

# ID del dataset
file_id = '1SFGqQuIdQWcQV7zS0vOE4td-8RZ575sS' 

# Construir la URL de descarga
url = f'https://drive.google.com/uc?id={file_id}'
extract_file = "Dataset.zip"

# Descargar el zip
gdown.download(url, extract_file, quiet=False)

print("ZIP descargado")

# Descomprimir
with zipfile.ZipFile(extract_file, 'r') as zip_ref:
    zip_ref.extractall(os.path.splitext(extract_file)[0])

os.remove(extract_file)
print("ZIP descomprimido en:", os.path.splitext(extract_file)[0])

In [ ]:
# Modelo Attention Res-UNet 2D para Audio Super Resolution

class AttentionGate(nn.Module):
    """Módulo de atención para ponderar skip connections."""
    def __init__(self, F_g, F_l, F_int):
        """
        Inicializa el módulo de atención.
        Args:
            F_g (int): Número de canales del gating signal.
            F_l (int): Número de canales de la skip connection.
            F_int (int): Número de canales intermedios.
        """
        super().__init__()

        # Uso de Attention Gates basado en el paper "https://arxiv.org/abs/1804.03999"
        self.W_g = nn.Conv2d(F_g, F_int, kernel_size=1)
        self.W_x = nn.Conv2d(F_l, F_int, kernel_size=1)
        self.psi = nn.Conv2d(F_int, 1, kernel_size=1)
        self.relu = nn.ReLU(inplace=True)
        self.sigmoid = nn.Sigmoid()

    def forward(self, g, x):
        # Interpolar g para que tenga el mismo tamaño que x
        if g.shape[-2:] != x.shape[-2:]:
            g = F.interpolate(g, size=x.shape[-2:], mode="bilinear", align_corners=False)
        # Calcular atención
        att = self.sigmoid(self.psi(self.relu(self.W_g(g) + self.W_x(x))))
        return x * att


class DilatedBlock(nn.Module):
    """Bloque con capas convolucionales dilatadas para capturar contexto de largo alcance."""
    def __init__(self, channels):
        """
        Inicializa el bloque dilatado.
        Args:
            channels (int): Número de canales de entrada y salida.
        """
        super().__init__()

        self.net = nn.Sequential(
            nn.Conv2d(channels, channels, kernel_size=(7,3), padding=(3,1), dilation=(1,1)),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(channels, channels, kernel_size=(7,3), padding=(6,2), dilation=(2,2)),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(channels, channels, kernel_size=(7,3), padding=(12,4), dilation=(4,4)),
            nn.LeakyReLU(0.2, inplace=True),
        )

    def forward(self, x):
        return self.net(x) + x


class ResBlock(nn.Module):
    """Bloque Residual para Attention Res-UNet."""
    def __init__(self, in_ch, out_ch):
        """
        Inicializa el bloque residual.
        Args:
            in_ch (int): Número de canales de entrada.
            out_ch (int): Número de canales de salida.
        """
        super().__init__()
        
        self.conv1 = nn.Conv2d(in_ch, out_ch, kernel_size=(7,3), padding=(3,1))
        self.norm1 = nn.GroupNorm(out_ch//4, out_ch) # GroupNorm para batchs pequeños
        self.relu1 = nn.LeakyReLU(0.2, inplace=True)
        
        self.conv2 = nn.Conv2d(out_ch, out_ch, kernel_size=(7,3), padding=(3,1))
        self.norm2 = nn.GroupNorm(out_ch//4, out_ch)
        self.relu2 = nn.LeakyReLU(0.2, inplace=True)
        
        # Skip connection
        if in_ch != out_ch:
            self.skip = nn.Sequential(
                nn.Conv2d(in_ch, out_ch, kernel_size=1, bias=False),
                nn.GroupNorm(out_ch//4, out_ch)
            )
        else:
            self.skip = nn.Identity()

    def forward(self, x):
        identity = self.skip(x)
        
        out = self.conv1(x)
        out = self.norm1(out)
        out = self.relu1(out)
        
        out = self.conv2(out)
        out = self.norm2(out)
        
        out += identity
        out = self.relu2(out)
        
        return out


class UNetAudio2D(nn.Module):
    """
    Modelo Attention Res-UNet 2D para super resolución de audio.
    Entrada y salida con shape (B, 2, F, T).
    """
    def __init__(self):
        """Inicializa la arquitectura UNet con encoder, bottleneck y decoder."""
        super().__init__()

        # Encoder
        # Entrada: (B, 2, F, T)
        self.enc1 = ResBlock(2, 32)
        self.down1 = nn.Conv2d(32, 32, kernel_size=(4,4), stride=(2,2), padding=(1,1))  # Strided conv

        self.enc2 = ResBlock(32, 64)
        self.down2 = nn.Conv2d(64, 64, kernel_size=(4,4), stride=(2,2), padding=(1,1))

        self.enc3 = ResBlock(64, 128)
        self.down3 = nn.Conv2d(128, 128, kernel_size=(4,4), stride=(2,2), padding=(1,1))

        self.enc4 = ResBlock(128, 256)
        self.down4 = nn.Conv2d(256, 256, kernel_size=(4,4), stride=(2,2), padding=(1,1))

        # Bottleneck
        self.bottleneck_conv = ResBlock(256, 512)
        self.bottleneck_dilated = DilatedBlock(512)

        # Decoder
        self.up4 = self.up_block(512,256)
        self.dec4 = ResBlock(512,256)

        self.up3 = self.up_block(256,128)
        self.dec3 = ResBlock(256,128)

        self.up2 = self.up_block(128,64)
        self.dec2 = ResBlock(128,64)

        self.up1 = self.up_block(64,32)
        self.dec1 = ResBlock(64,32)

        # Attention gates
        self.att4 = AttentionGate(256,256,128)
        self.att3 = AttentionGate(128,128,64)
        self.att2 = AttentionGate(64,64,32)
        self.att1 = AttentionGate(32,32,16)

        # Output
        self.final = nn.Conv2d(32,2,kernel_size=1)

    def up_block(self, in_ch, out_ch):
        """
        Crea un bloque de upsampling con ConvTranspose2d.
        Args:
            in_ch (int): Número de canales de entrada.
            out_ch (int): Número de canales de salida.
        Returns:
            nn.Sequential: Bloque de upsampling en frecuencia y tiempo.
        """
        return nn.Sequential(
            nn.ConvTranspose2d(in_ch, out_ch, kernel_size=(4,4), stride=(2,2), padding=(1,1)),
            nn.LeakyReLU(0.2, inplace=True)
        )

    def forward(self, x):
        # Encoder
        e1 = self.enc1(x)
        e2 = self.enc2(self.down1(e1))
        e3 = self.enc3(self.down2(e2))
        e4 = self.enc4(self.down3(e3))

        # Bottleneck
        b = self.bottleneck_conv(self.down4(e4))
        b = self.bottleneck_dilated(b)

        # Decoder con skip connections y attention gates
        up4 = self.up4(b)
        d4 = self.dec4(torch.cat([up4, self.att4(up4, e4)], dim=1))

        up3 = self.up3(d4)
        d3 = self.dec3(torch.cat([up3, self.att3(up3, e3)], dim=1))

        up2 = self.up2(d3)
        d2 = self.dec2(torch.cat([up2, self.att2(up2, e2)], dim=1))

        up1 = self.up1(d2)
        d1 = self.dec1(torch.cat([up1, self.att1(up1, e1)], dim=1))

        return self.final(d1) + x

In [ ]:
# Discriminador Multi-Escala (MSD) y Multi-Periodo (MPD)
# Basado en HiFi-GAN
# https://arxiv.org/pdf/2010.05646


from torch.nn.utils.parametrizations import weight_norm, spectral_norm


def get_padding(kernel_size, dilation=1):
    return int((kernel_size*dilation - dilation)/2)

class DiscriminatorP(torch.nn.Module):
    def __init__(self, period, kernel_size=5, stride=3, use_spectral_norm=False):
        super(DiscriminatorP, self).__init__()
        self.period = period
        norm_f = weight_norm if use_spectral_norm == False else spectral_norm
        self.convs = nn.ModuleList([
            norm_f(nn.Conv2d(1, 32, (kernel_size, 1), (stride, 1), padding=(get_padding(kernel_size, 1), 0))),
            norm_f(nn.Conv2d(32, 128, (kernel_size, 1), (stride, 1), padding=(get_padding(kernel_size, 1), 0))),
            norm_f(nn.Conv2d(128, 512, (kernel_size, 1), (stride, 1), padding=(get_padding(kernel_size, 1), 0))),
            norm_f(nn.Conv2d(512, 1024, (kernel_size, 1), (stride, 1), padding=(get_padding(kernel_size, 1), 0))),
            norm_f(nn.Conv2d(1024, 1024, (kernel_size, 1), 1, padding=(get_padding(kernel_size, 1), 0))),
        ])
        self.conv_post = norm_f(nn.Conv2d(1024, 1, (3, 1), 1, padding=(1, 0)))

    def forward(self, x):
        fmap = []

        # 1d to 2d
        b, c, t = x.shape
        if t % self.period != 0: # pad first
            n_pad = self.period - (t % self.period)
            x = F.pad(x, (0, n_pad), "reflect")
            t = t + n_pad
        x = x.view(b, c, t // self.period, self.period)

        for l in self.convs:
            x = l(x)
            x = F.leaky_relu(x, 0.1)
            fmap.append(x)
        x = self.conv_post(x)
        fmap.append(x)
        x = torch.flatten(x, 1, -1)

        return x, fmap


class MultiPeriodDiscriminator(torch.nn.Module):
    def __init__(self):
        super(MultiPeriodDiscriminator, self).__init__()
        self.discriminators = nn.ModuleList([
            DiscriminatorP(2),
            DiscriminatorP(3),
            DiscriminatorP(5),
            DiscriminatorP(7),
            DiscriminatorP(11),
        ])

    def forward(self, y, y_hat):
        y_d_rs = []
        y_d_gs = []
        fmap_rs = []
        fmap_gs = []
        for i, d in enumerate(self.discriminators):
            y_d_r, fmap_r = d(y)
            y_d_g, fmap_g = d(y_hat)
            y_d_rs.append(y_d_r)
            fmap_rs.append(fmap_r)
            y_d_gs.append(y_d_g)
            fmap_gs.append(fmap_g)

        return y_d_rs, y_d_gs, fmap_rs, fmap_gs


class DiscriminatorS(torch.nn.Module):
    def __init__(self, use_spectral_norm=False):
        super(DiscriminatorS, self).__init__()
        norm_f = weight_norm if use_spectral_norm == False else spectral_norm
        self.convs = nn.ModuleList([
            norm_f(nn.Conv1d(1, 128, 15, 1, padding=7)),
            norm_f(nn.Conv1d(128, 128, 41, 2, groups=4, padding=20)),
            norm_f(nn.Conv1d(128, 256, 41, 2, groups=16, padding=20)),
            norm_f(nn.Conv1d(256, 512, 41, 4, groups=16, padding=20)),
            norm_f(nn.Conv1d(512, 1024, 41, 4, groups=16, padding=20)),
            norm_f(nn.Conv1d(1024, 1024, 41, 1, groups=16, padding=20)),
            norm_f(nn.Conv1d(1024, 1024, 5, 1, padding=2)),
        ])
        self.conv_post = norm_f(nn.Conv1d(1024, 1, 3, 1, padding=1))

    def forward(self, x):
        fmap = []
        for l in self.convs:
            x = l(x)
            x = F.leaky_relu(x, 0.1)
            fmap.append(x)
        x = self.conv_post(x)
        fmap.append(x)
        x = torch.flatten(x, 1, -1)

        return x, fmap


class MultiScaleDiscriminator(torch.nn.Module):
    def __init__(self):
        super(MultiScaleDiscriminator, self).__init__()
        self.discriminators = nn.ModuleList([
            DiscriminatorS(use_spectral_norm=True),
            DiscriminatorS(),
            DiscriminatorS(),
        ])
        self.meanpools = nn.ModuleList([
            nn.AvgPool1d(4, 2, padding=2),nn.AvgPool1d(4, 2, padding=2)
        ])

    def forward(self, y, y_hat):
        y_d_rs = []
        y_d_gs = []
        fmap_rs = []
        fmap_gs = []
        for i, d in enumerate(self.discriminators):
            if i != 0:
                y = self.meanpools[i-1](y)
                y_hat = self.meanpools[i-1](y_hat)
            y_d_r, fmap_r = d(y)
            y_d_g, fmap_g = d(y_hat)
            y_d_rs.append(y_d_r)
            fmap_rs.append(fmap_r)
            y_d_gs.append(y_d_g)
            fmap_gs.append(fmap_g)

        return y_d_rs, y_d_gs, fmap_rs, fmap_gs

class CombinedDiscriminator(nn.Module):
    """
    Módulo del discriminador que combina MSD y MPD.
    """
    def __init__(self):
        super().__init__()
        
        self.msd = MultiScaleDiscriminator()
        self.mpd = MultiPeriodDiscriminator()

    def forward(self, y, y_hat):
        y_d_rs, y_d_gs, fmap_rs, fmap_gs = [], [], [], []
        
        for d in [self.msd, self.mpd]:
            y_r, y_g, fmap_r, fmap_g = d(y, y_hat)
            y_d_rs.extend(y_r)
            y_d_gs.extend(y_g)
            fmap_rs.extend(fmap_r)
            fmap_gs.extend(fmap_g)
            
        return y_d_rs, y_d_gs, fmap_rs, fmap_gs

In [ ]:
# Script para cargar el dataset
# Devuelve pares de STFT (real + imaginario) con shape (2, F, T) en fragmentos de 1.5s

NFFT = 1024
HOP_LENGTH = 256
SAMPLE_RATE = 44100
FRAGMENT_LENGTH = 65280

class AudioSuperResDataset(Dataset):
    """
    Carga pares de audio en Alta Calidad (HR) y Baja Calidad (LR),
    los divide en fragmentos de FRAGMENT_LENGTH muestras y devuelve
    sus representaciones STFT con shape (2, F, T).
    """
    def __init__(self, hr_dir, lr_dir, fragment_length=FRAGMENT_LENGTH, n_fft=NFFT, hop_length=HOP_LENGTH):
        """
        Inicializa el dataset cargando y fragmentando los archivos de audio.
        Args:
            hr_dir (str): Directorio de los archivos en alta calidad (Ground Truth).
            lr_dir (str): Directorio de los archivos en baja calidad (Input).
            fragment_length (int): Longitud del fragmento a procesar en muestras.
            n_fft (int): Tamaño de la FFT para la STFT.
            hop_length (int): Salto entre ventanas STFT.
        """

        self.hr_dir = hr_dir
        self.lr_dir = lr_dir
        self.fragment_length = fragment_length
        self.n_fft = n_fft
        self.hop_length = hop_length

        # POOL_FACTOR = 2^4 = 16 (4 capas de pooling en la UNet 2D)
        self.pool_factor = 16

        # Cache de resamplers
        self._resamplers = {}

        # Ventana para STFT
        self.window = torch.hann_window(self.n_fft)

        # Contar ficheros
        files = [
            f for f in os.listdir(hr_dir)
            if f.endswith('.wav') and os.path.exists(os.path.join(lr_dir, f))
        ]

        # Cargar ficheros y dividir en fragmentos
        self.fragments = []  # lista (hr_fragment, lr_fragment)

        for filename in files:
            hr_path = os.path.join(hr_dir, filename)
            lr_path = os.path.join(lr_dir, filename)

            # Extraer metadatos
            info_hr = torchaudio.info(hr_path)
            info_lr = torchaudio.info(lr_path)

            sr_hr = info_hr.sample_rate
            sr_lr = info_lr.sample_rate

            # Calcular número de muestras originales correspondientes a FRAGMENT_LENGTH
            frag_len_hr_orig = int(self.fragment_length * (sr_hr / SAMPLE_RATE))
            frag_len_lr_orig = int(self.fragment_length * (sr_lr / SAMPLE_RATE))

            num_fragments = min(info_hr.num_frames // frag_len_hr_orig, info_lr.num_frames // frag_len_lr_orig)

            for i in range(num_fragments):
                self.fragments.append({
                    'filename': filename,
                    'start_hr': i * frag_len_hr_orig,
                    'start_lr': i * frag_len_lr_orig,
                    'frames_hr': frag_len_hr_orig,
                    'frames_lr': frag_len_lr_orig,
                    'sr_hr': sr_hr,
                    'sr_lr': sr_lr
                })

    def __len__(self):
        """Devuelve el número de fragmentos del conjunto."""
        return len(self.fragments)

    def _waveform_to_stft(self, waveform):
        """Convierte una waveform (1, L) a un tensor STFT de shape (2, F, T).
        El canal 0 corresponde a la parte real y el canal 1 a la parte imaginaria.
        Args:
            waveform (torch.Tensor): Tensor de audio de entrada con shape (1, L).
        Returns:
            torch.Tensor: Tensor STFT con partes real e imaginaria apiladas con shape (2, F, T).
        """
        stft = torch.stft(
            waveform.squeeze(0),          #(L,)
            n_fft=self.n_fft,
            hop_length=self.hop_length,
            win_length=self.n_fft,
            window=self.window,
            return_complex=True,
        )  #stft shape: (F, T)

        # Apilar parte real e imaginaria
        stft_ri = torch.stack([stft.real, stft.imag], dim=0) # (2, F, T)
        return stft_ri

    def _normalize_stft(self, stft_ri):
        """
        Compresión logarítmica de la magnitud de la STFT preservando la fase.
        Args:
            stft_ri (torch.Tensor): Tensor STFT real/imaginario de dimensiones (2, F, T).
        Returns:
            torch.Tensor: Tensor comprimido (2, F, T).
        """
        real = stft_ri[0]
        imag = stft_ri[1]
        
        magnitude = torch.sqrt(real**2 + imag**2 + 1e-8)
        phase_cos = real / magnitude  # cos(phase)
        phase_sin = imag / magnitude  # sin(phase)
        
        mag_compressed = torch.log1p(magnitude)
        
        return torch.stack([mag_compressed * phase_cos, mag_compressed * phase_sin], dim=0)

    def _pad_stft_to_pool_factor(self, stft, pool_factor=16):
        """
        Aplica padding a STFT para que F y T sean divisibles por pool_factor.
        Args:
            stft (torch.Tensor): Tensor STFT (2, F, T).
            pool_factor (int): Factor de pooling.
        Returns:
            torch.Tensor: Tensor STFT con dimensiones divisibles por pool_factor.
        """
        _, freq_bins, time_frames = stft.shape

        # Pad frecuencia
        pad_f = (pool_factor - (freq_bins % pool_factor)) % pool_factor
        # Pad tiempo
        pad_t = (pool_factor - (time_frames % pool_factor)) % pool_factor

        if pad_f > 0 or pad_t > 0:
            stft = F.pad(stft, (0, pad_t, 0, pad_f), mode='reflect')    # Padding mediante reflexión

        return stft

    def __getitem__(self, idx):
        # Cargar fragmento
        frag_info = self.fragments[idx]
        hr_path = os.path.join(self.hr_dir, frag_info['filename'])
        lr_path = os.path.join(self.lr_dir, frag_info['filename'])

        # Leer solo el trozo necesario
        frag_hr, _ = torchaudio.load(hr_path, frame_offset=frag_info['start_hr'], num_frames=frag_info['frames_hr'])
        frag_lr, _ = torchaudio.load(lr_path, frame_offset=frag_info['start_lr'], num_frames=frag_info['frames_lr'])

        # Uniformizar a mono
        if frag_hr.size(0) > 1:
            frag_hr = frag_hr.mean(dim=0, keepdim=True)
        if frag_lr.size(0) > 1:
            frag_lr = frag_lr.mean(dim=0, keepdim=True)

        # Resamplear a 44.1 kHz si es necesario
        if frag_info['sr_hr'] != SAMPLE_RATE:
            key = ('hr', frag_info['sr_hr'])
            if key not in self._resamplers:
                self._resamplers[key] = torchaudio.transforms.Resample(frag_info['sr_hr'], SAMPLE_RATE)
            frag_hr = self._resamplers[key](frag_hr)

        if frag_info['sr_lr'] != SAMPLE_RATE:
            key = ('lr', frag_info['sr_lr'])
            if key not in self._resamplers:
                self._resamplers[key] = torchaudio.transforms.Resample(frag_info['sr_lr'], SAMPLE_RATE)
            frag_lr = self._resamplers[key](frag_lr)

        # Forzar longitud exacta
        if frag_hr.size(1) != self.fragment_length:
            frag_hr = F.pad(frag_hr, (0, max(0, self.fragment_length - frag_hr.size(1))))[:, :self.fragment_length]
        if frag_lr.size(1) != self.fragment_length:
            frag_lr = F.pad(frag_lr, (0, max(0, self.fragment_length - frag_lr.size(1))))[:, :self.fragment_length]

        # Descartar silencios
        if frag_hr.abs().max() < 1e-4:
            return self.__getitem__((idx + 1) % len(self))

        # Normalizar a [-1, 1]
        max_val = max(frag_hr.abs().max(), frag_lr.abs().max()) + 1e-8
        frag_lr = frag_lr / max_val
        frag_hr = frag_hr / max_val

        stft_lr = self._waveform_to_stft(frag_lr)
        stft_hr = self._waveform_to_stft(frag_hr)

        # Normalizar STFT con log-compresión
        stft_lr = self._normalize_stft(stft_lr)
        stft_hr = self._normalize_stft(stft_hr)

        # Asegurar que F y T son divisibles por pool_factor (16)
        stft_lr = self._pad_stft_to_pool_factor(stft_lr, self.pool_factor)
        stft_hr = self._pad_stft_to_pool_factor(stft_hr, self.pool_factor)

        # Devolver STFT: Input LR y Target HR, ambos con shape (2, F, T)
        return stft_lr, stft_hr

In [ ]:
# Loss para Audio Super-Resolución y el Discriminador
# Este script contiene las funciones de pérdida para entrenar la red neuronal, calculando
# la pérdida en el dominio del tiempo y en el dominio de la frecuencia.
# Loss de validación basado en la Loss implementada en AERO: https://github.com/slp-rl/aero
# Loss de entrenamiento basado en la Loss implementada en HiFiGAN: https://github.com/jik876/hifi-gan

from torchaudio.transforms import MelSpectrogram
from torchmetrics.audio import ScaleInvariantSignalDistortionRatio
from torchmetrics.audio.stoi import ShortTimeObjectiveIntelligibility

NFFT = 1024
HOP_LENGTH = 256
WIN_LENGTH = 1024
SAMPLE_RATE = 44100
FRAGMENT_LENGTH = 65280

class STFTLoss(nn.Module):
    """Pérdida de Magnitud Logarítmica y Convergencia Espectral (STFT)."""
    def __init__(self, n_fft=NFFT, hop_length=HOP_LENGTH, win_length=WIN_LENGTH):
        """
        Inicializa los parámetros de la STFT.
        Args:
            n_fft (int): Tamaño de la FFT.
            hop_length (int): Salto entre ventanas.
            win_length (int): Tamaño de la ventana.
        """
        super().__init__()

        self.n_fft = n_fft
        self.hop_length = hop_length
        self.win_length = win_length
        self.register_buffer("window", torch.hann_window(win_length))

    def forward(self, x, y):
        # x, y shape: (B, T)
        x_stft = torch.stft(x, self.n_fft, self.hop_length, self.win_length, self.window, return_complex=True)
        y_stft = torch.stft(y, self.n_fft, self.hop_length, self.win_length, self.window, return_complex=True)

        x_mag = torch.abs(x_stft)
        y_mag = torch.abs(y_stft)

        # Spectral Convergence Loss
        sc_loss = torch.norm(y_mag - x_mag, p="fro") / (torch.norm(y_mag, p="fro") + 1e-8)
        
        # Log-Magnitude Loss
        mag_loss = F.l1_loss(torch.log(x_mag + 1e-8), torch.log(y_mag + 1e-8))

        return sc_loss, mag_loss


class MultiResolutionSTFTLoss(nn.Module):
    """Pérdida STFT multi-resolución basada en AERO."""
    def __init__(self, fft_sizes=[1024, 2048, 512], hop_sizes=[120, 240, 50], win_lengths=[600, 1200, 240]):
        """
        Inicializa las pérdidas STFT para múltiples resoluciones.
        Args:
            fft_sizes (list[int]): Tamaños de FFT para cada resolución.
            hop_sizes (list[int]): Saltos entre ventanas para cada resolución.
            win_lengths (list[int]): Tamaños de ventana para cada resolución.
        """
        super().__init__()

        assert len(fft_sizes) == len(hop_sizes) == len(win_lengths)
        
        # STFT diferentes resoluciones
        self.stft_losses = nn.ModuleList()
        for fs, ss, wl in zip(fft_sizes, hop_sizes, win_lengths):
            self.stft_losses.append(STFTLoss(fs, ss, wl))

    def forward(self, x, y):
        sc_loss = 0.0
        mag_loss = 0.0
        
        for f in self.stft_losses:
            sc_l, mag_l = f(x, y)
            sc_loss += sc_l
            mag_loss += mag_l
        
        sc_loss /= len(self.stft_losses)
        mag_loss /= len(self.stft_losses)
        
        return sc_loss + mag_loss


class CombinedLoss(nn.Module):
    """Pérdida combinada L1 temporal y Multi-Resolución STFT."""
    def __init__(self, n_fft=NFFT, hop_length=HOP_LENGTH, lambda_l1=1.0, lambda_mrstft=1.0):
        """
        Inicializa la pérdida combinada.
        Args:
            n_fft (int): Tamaño de la FFT.
            hop_length (int): Salto entre ventanas.
            lambda_l1 (float): Peso de la pérdida L1.
            lambda_mrstft (float): Peso de la pérdida MRSTFT.
        """
        super().__init__()

        self.n_fft = n_fft
        self.hop_length = hop_length
        
        self.lambda_l1 = lambda_l1
        self.lambda_mrstft = lambda_mrstft
        
        # Buffer ISTFT
        self.register_buffer('base_window', torch.hann_window(n_fft))
        
        # MRSTFT
        self.mrstft_loss = MultiResolutionSTFTLoss()

    def _denormalize_stft(self, stft_ri):
        """
        Inversa de la compresión logarítmica de la magnitud.
        Args:
            stft_ri (torch.Tensor): Tensor STFT comprimido (B, 2, F, T).
        Returns:
            torch.Tensor: Tensor STFT en escala lineal (B, 2, F, T).
        """
        real = stft_ri[:, 0]
        imag = stft_ri[:, 1]
        
        mag_compressed = torch.sqrt(real**2 + imag**2 + 1e-8)
        phase_cos = real / mag_compressed
        phase_sin = imag / mag_compressed
        
        magnitude = torch.exp(mag_compressed) - 1
        
        return torch.stack([magnitude * phase_cos, magnitude * phase_sin], dim=1)

    def _stft_to_waveform(self, stft_ri):
        """
        Aplica la Inversa del STFT (ISTFT) para recuperar la forma de onda.
        Args:
            stft_ri (torch.Tensor): Tensor STFT real/imaginario.
        Returns:
            torch.Tensor: Tensor de la forma de onda reconstruida.
        """
        # Deshacer padding en frecuencia
        valid_freq_bins = self.n_fft // 2 + 1
        if stft_ri.size(2) > valid_freq_bins:
            stft_ri = stft_ri[:, :, :valid_freq_bins, :]
            
        # Deshacer padding en tiempo
        valid_time_frames = FRAGMENT_LENGTH // self.hop_length + 1
        if stft_ri.size(3) > valid_time_frames:
            stft_ri = stft_ri[:, :, :, :valid_time_frames]

        stft_complex = torch.complex(stft_ri[:, 0], stft_ri[:, 1])
        waveform = torch.istft(
            stft_complex,
            n_fft=self.n_fft,
            hop_length=self.hop_length,
            win_length=self.n_fft,
            window=self.base_window,
            length=FRAGMENT_LENGTH
        )
        return waveform

    def get_waveforms(self, pred_stft, target_stft):
        """
        Pasa de representaciones STFT log-comprimidas a formas de onda en el tiempo.
        Args:
            pred_stft (torch.Tensor): STFT predicción del modelo.
            target_stft (torch.Tensor): STFT objetivo del modelo.
        Returns:
            tuple[torch.Tensor, torch.Tensor]: Las formas de onda (B, 1, T)
                para predicción y objetivo.
        """
        # Deshacer compresión logarítmica
        pred_linear = self._denormalize_stft(pred_stft)
        target_linear = self._denormalize_stft(target_stft)
        
        # Reconstrucción a waveform
        pred_wav = self._stft_to_waveform(pred_linear)
        target_wav = self._stft_to_waveform(target_linear)
    
        # Recortar a la misma longitud mínima
        min_len = min(pred_wav.size(-1), target_wav.size(-1))
        pred_wav = pred_wav[..., :min_len]
        target_wav = target_wav[..., :min_len]
    
        # Asegurar shape (B, 1, T) para el discriminador 1D
        if pred_wav.ndim == 2:
            pred_wav = pred_wav.unsqueeze(1)
            target_wav = target_wav.unsqueeze(1)
        
        return pred_wav, target_wav

    def forward(self, pred_stft, target_stft, pred_wav=None, target_wav=None):
        # Si no se proporcionan waveforms precalculadas, calcular internamente
        if pred_wav is None or target_wav is None:
            # Descomprimir
            pred_linear = self._denormalize_stft(pred_stft)
            target_linear = self._denormalize_stft(target_stft)
            
            # Reconstrucción a Waveform
            pred_wav = self._stft_to_waveform(pred_linear)
            target_wav = self._stft_to_waveform(target_linear)
        
        # Recortar a longitud mínima
        min_len = min(pred_wav.size(-1), target_wav.size(-1))
        pred_wav = pred_wav[..., :min_len]
        target_wav = target_wav[..., :min_len]
        
        # L1 loss
        loss_l1 = F.l1_loss(pred_wav, target_wav)
        
        # MRSTFT loss
        if pred_wav.ndim == 3:
            pred_wav = pred_wav.squeeze(1)
            target_wav = target_wav.squeeze(1)
        loss_mrstft = self.mrstft_loss(pred_wav, target_wav)
        
        return self.lambda_l1 * loss_l1 + self.lambda_mrstft * loss_mrstft


# Pérdidas del discriminador
class DiscriminatorLoss(nn.Module):
    """Pérdidas del discriminador."""
    def __init__(self, sample_rate=SAMPLE_RATE, n_fft=NFFT, hop_length=HOP_LENGTH, win_length=WIN_LENGTH, lambda_adv=1.0, lambda_fm=2.0, lambda_mel=45.0, n_mels=80):
        """
        Inicializa las pérdidas del discriminador.
        Args:
            sample_rate (int): Frecuencia de muestreo del audio.
            n_fft (int): Tamaño de la FFT.
            hop_length (int): Salto entre ventanas.
            win_length (int): Tamaño de la ventana.
            lambda_adv (float): Peso de la pérdida adversarial.
            lambda_fm (float): Peso del feature matching.
            lambda_mel (float): Peso de la pérdida Mel.
            n_mels (int): Número de bandas Mel.
        """
        super().__init__()

        self.lambda_adv = lambda_adv
        self.lambda_fm = lambda_fm
        self.lambda_mel = lambda_mel

        self.mel_spec = MelSpectrogram(
            sample_rate=sample_rate,
            n_fft=n_fft,
            hop_length=hop_length,
            win_length=win_length,
            n_mels=n_mels,
            center=True,
            power=1.0,
            norm="slaney",
            mel_scale="slaney",
        )

    def discriminator_loss(self, disc_real_outputs, disc_generated_outputs):
        """
        Pérdida LSGAN del discriminador.
        Args:
            disc_real_outputs (list[torch.Tensor]): Salida del discriminador para inputs reales.
            disc_generated_outputs (list[torch.Tensor]): Salida para inputs generados.
        Returns:
            torch.Tensor: Valor de la pérdida total del discriminador.
        """
        loss = 0
        for dr, dg in zip(disc_real_outputs, disc_generated_outputs):
            dr_loss = torch.mean((1 - dr)**2)
            dg_loss = torch.mean(dg**2)
            loss += (dr_loss + dg_loss)
        return self.lambda_adv * loss

    def generator_loss(self, disc_generated_outputs):
        """
        Pérdida LSGAN del generador.
        Args:
            disc_generated_outputs (list[torch.Tensor]): Salida del discriminador para inputs generados.
        Returns:
            torch.Tensor: Valor de la pérdida adversarial del generador.
        """
        loss = 0
        for dg in disc_generated_outputs:
            loss += torch.mean((1 - dg)**2)
        return self.lambda_adv * loss

    def feature_matching_loss(self, fmap_r, fmap_g):
        """
        Pérdida L1 entre los feature maps del discriminador.
        Args:
            fmap_r (list[list[torch.Tensor]]): Mapa de características sobre el audio real.
            fmap_g (list[list[torch.Tensor]]): Mapa de características sobre el audio generado.
        Returns:
            torch.Tensor: Valor acumulado de las diferencias L1.
        """
        loss = 0
        for dr, dg in zip(fmap_r, fmap_g):
            for rl, gl in zip(dr, dg):
                loss += torch.mean(torch.abs(rl - gl))
        return self.lambda_fm * loss

    def mel_spectrogram_loss(self, pred_wav, target_wav):
        """
        Pérdida L1 entre los Mel-spectrogramas del predictor y el objetivo.
        Args:
            pred_wav (torch.Tensor): Waveform predicción del modelo.
            target_wav (torch.Tensor): Waveform objetivo del modelo.
        Returns:
            torch.Tensor: Valor de la pérdida L1 entre los Mel-spectrogramas.
        """
        if pred_wav.ndim == 3:
            pred_wav = pred_wav.squeeze(1)
        if target_wav.ndim == 3:
            target_wav = target_wav.squeeze(1)

        pred_mel = torch.log(self.mel_spec(pred_wav) + 1e-8)
        target_mel = torch.log(self.mel_spec(target_wav) + 1e-8)
        mel_loss = F.l1_loss(pred_mel, target_mel)

        return self.lambda_mel * mel_loss


class LossMetrics(nn.Module):
    """Métricas de calidad de audio."""
    @staticmethod
    def sisdr_loss(pred_wav, target_wav):
        """Pérdida SI-SDR."""
        if pred_wav.ndim == 3 and pred_wav.size(1) == 1:
            pred_wav = pred_wav.squeeze(1)
            target_wav = target_wav.squeeze(1)
        sisdr = ScaleInvariantSignalDistortionRatio().to(pred_wav.device)
        return sisdr(pred_wav, target_wav)
    
    @staticmethod
    def stoi_loss(pred_wav, target_wav):
        """Pérdida STOI."""
        if pred_wav.ndim == 3 and pred_wav.size(1) == 1:
            pred_wav = pred_wav.squeeze(1)
            target_wav = target_wav.squeeze(1)
        stoi = ShortTimeObjectiveIntelligibility(SAMPLE_RATE).to(pred_wav.device)
        return stoi(pred_wav, target_wav)

    @staticmethod
    def lsd_loss(pred_wav, target_wav, n_fft=NFFT, hop_length=HOP_LENGTH, eps=1e-8):
        """Pérdida LSD"""
        window = torch.hann_window(n_fft).to(pred_wav.device)

        if pred_wav.ndim == 3 and pred_wav.size(1) == 1:
            pred_wav = pred_wav.squeeze(1)
            target_wav = target_wav.squeeze(1)
        
        Sx = torch.abs(torch.stft(pred_wav, n_fft, hop_length, window=window, return_complex=True))
        Sy = torch.abs(torch.stft(target_wav, n_fft, hop_length, window=window, return_complex=True))
        lsd = torch.mean((torch.log(Sx + eps) - torch.log(Sy + eps)) ** 2, dim=(-2, -1))
        return torch.mean(torch.sqrt(lsd)).item()

In [ ]:
# Script para entrenar la red neuronal
# Este script guardara el mejor modelo en un archivo .pt

import copy
from torch.utils.tensorboard import SummaryWriter

TRAIN_HR_DIR = './Dataset/train/HR'    # Archivos de alta resolución (ground truth)
TRAIN_LR_DIR = './Dataset/train/LR'    # Archivos de baja resolución (input)
VAL_HR_DIR = './Dataset/test/HR'       # Archivos de alta resolución para validación
VAL_LR_DIR = './Dataset/test/LR'       # Archivos de baja resolución para validación

BATCH_SIZE = 8                      # Tamaño de lote
EPOCHS = 500                        # Épocas
LEARNING_RATE_G = 2e-4              # LR del generador
LEARNING_RATE_D = 1e-4              # LR del discriminador

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

def evaluate(model_g, dataloader, criterion):
    """Evalúa el modelo en el conjunto de validación y devuelve la pérdida promedio, SI-SDR, STOI y LSD."""
    model_g.eval()
    total_loss = 0.0
    total_sisdr = 0.0
    total_stoi = 0.0
    total_lsd = 0.0
    with torch.no_grad():
        # Validar el modelo sin gradientes
        for inputs, targets in dataloader:
            inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)
            outputs = model_g(inputs)   # Predicción del modelo
            loss = criterion(outputs, targets)  # Pérdida del modelo
            total_loss += loss.item()
            pred_wav, target_wav = criterion.get_waveforms(outputs, targets) # Obtener waveforms
            total_sisdr += LossMetrics.sisdr_loss(pred_wav, target_wav) # SI-SDR
            total_stoi += LossMetrics.stoi_loss(pred_wav, target_wav) # STOI
            total_lsd += LossMetrics.lsd_loss(pred_wav, target_wav) # LSD
    return total_loss / len(dataloader), total_sisdr / len(dataloader), total_stoi / len(dataloader), total_lsd / len(dataloader)

def train():
    """Entrenar el modelo"""
    # Verificar que los directorios existen
    dirs_to_check = [TRAIN_HR_DIR, TRAIN_LR_DIR, VAL_HR_DIR, VAL_LR_DIR]
    for d in dirs_to_check:
        if not os.path.exists(d):
            print(f"Error: directorio no encontrado: {d}")
            return

    # Cargar datasets
    train_dataset = AudioSuperResDataset(TRAIN_HR_DIR, TRAIN_LR_DIR)
    val_dataset = AudioSuperResDataset(VAL_HR_DIR, VAL_LR_DIR)

    if len(train_dataset) == 0 or len(val_dataset) == 0:
        print("Error: El dataset está vacío")
        return

    num_workers = max(0, os.cpu_count()-1)
    # len(dataset) -> número de fragmentos del conjunto
    # Shape: (2, F, T) -> 2=componentes real/imag, F=frecuencia, T=tiempo
    print(f"Cargados {len(train_dataset)} elementos de entrenamiento con shapes de entrada y salida: {train_dataset[0][0].shape} y {train_dataset[0][1].shape}")
    train_dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=num_workers, pin_memory=True)
    print(f"Cargados {len(val_dataset)} elementos de validación con shapes de entrada y salida: {val_dataset[0][0].shape} y {val_dataset[0][1].shape}")
    val_dataloader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=num_workers, pin_memory=True)

    # Inicializar Modelos
    model_g = UNetAudio2D().to(DEVICE)
    model_d = CombinedDiscriminator().to(DEVICE)

    # Inicializar EMA en generador
    ema_decay = 0.999
    ema_model = copy.deepcopy(model_g)
    ema_model.eval()
    for p in ema_model.parameters():
        p.requires_grad_(False)

    # Inicializar Pérdidas
    criterion_g = CombinedLoss(lambda_l1=1.0, lambda_mrstft=1.0).to(DEVICE)
    criterion_d = DiscriminatorLoss(lambda_adv=1.0, lambda_fm=2.0, lambda_mel=45.0).to(DEVICE)

    # Inicializar Optimizadores
    optimizer_g = optim.AdamW(model_g.parameters(), lr=LEARNING_RATE_G, betas=(0.8, 0.99))
    optimizer_d = optim.AdamW(model_d.parameters(), lr=LEARNING_RATE_D, betas=(0.8, 0.99))

    # Schedulers
    scheduler_g = optim.lr_scheduler.ReduceLROnPlateau(optimizer_g, mode='min', factor=0.5, patience=10)
    scheduler_d = optim.lr_scheduler.ReduceLROnPlateau(optimizer_d, mode='min', factor=0.5, patience=10)

    # Inicializar TensorBoard
    writer = SummaryWriter(log_dir='./runs')

    print(f"Iniciando entrenamiento en {DEVICE}...")

    # Early stopping
    best_val_loss = float('inf')
    patience_earlystop = 50
    epochs_no_improve = 0

    # Bucle de entrenamiento
    for epoch in range(EPOCHS):
        # Entrenar modelos
        model_g.train()
        model_d.train()

        running_loss_g = 0.0
        running_loss_d = 0.0

        for i, (inputs, targets) in enumerate(train_dataloader):
            inputs = inputs.to(DEVICE)
            targets = targets.to(DEVICE)
            
            # Entrenar generador
            pred = model_g(inputs)

            # Obtener waveforms de LR y HR
            pred_wav, target_wav = criterion_g.get_waveforms(pred, targets)

            # Entrenar discriminador
            optimizer_d.zero_grad(set_to_none=True)
            y_d_rs, y_d_gs, _, _ = model_d(target_wav.detach(), pred_wav.detach())
            loss_d = criterion_d.discriminator_loss(y_d_rs, y_d_gs)

            # Backward
            loss_d.backward()

            # Gradient clipping para evitar explosión de gradientes
            torch.nn.utils.clip_grad_norm_(model_d.parameters(), max_norm=1.0)

            # Actualizar los pesos del modelo
            optimizer_d.step()

            # Acumular pérdidas
            running_loss_d += loss_d.item()

            optimizer_g.zero_grad(set_to_none=True)
            
            # Perdidas del generador
            y_d_rs, y_d_gs, fmap_rs, fmap_gs = model_d(target_wav.detach(), pred_wav)
            loss_adv = criterion_d.generator_loss(y_d_gs)
            loss_fm = criterion_d.feature_matching_loss(fmap_rs, fmap_gs)
            loss_mel = criterion_d.mel_spectrogram_loss(pred_wav, target_wav.detach())
            loss_g = loss_adv + loss_fm + loss_mel
            
            # Backward
            loss_g.backward()
            
            # Gradient clipping para evitar explosión de gradientes
            torch.nn.utils.clip_grad_norm_(model_g.parameters(), max_norm=1.0)
            
            # Actualizar los pesos del modelo
            optimizer_g.step()

            # Actualizar EMA del generador
            with torch.no_grad():
                for ema_p, model_p in zip(ema_model.parameters(), model_g.parameters()):
                    ema_p.data.mul_(ema_decay).add_(model_p.data, alpha=1.0 - ema_decay)

            # Acumular pérdidas
            running_loss_g += loss_g.item()
        
        # Métricas
        train_loss_g = running_loss_g / len(train_dataloader)
        train_loss_d = running_loss_d / len(train_dataloader)

        # Evaluar generador en el conjunto de validación
        val_loss, val_sisdr, val_stoi, val_lsd = evaluate(ema_model, val_dataloader, criterion_g)

        print(f"Epoch [{epoch+1}/{EPOCHS}] | Loss G: {train_loss_g:.6f} | Loss D: {train_loss_d:.6f} | Val Loss: {val_loss:.6f} | LR: {optimizer_g.param_groups[0]['lr']:.8f}")
        print(f"Métricas de calidad: SI-SDR: {val_sisdr:.6f} | STOI: {val_stoi:.6f} | LSD: {val_lsd:.6f}")

        # Registrar métricas en TensorBoard
        writer.add_scalars('Loss/train', {'generator': train_loss_g, 'discriminator': train_loss_d}, epoch)
        writer.add_scalar('Loss/val', val_loss, epoch)
        writer.add_scalar('Metrics/SI-SDR', val_sisdr, epoch)
        writer.add_scalar('Metrics/STOI', val_stoi, epoch)
        writer.add_scalar('Metrics/LSD', val_lsd, epoch)
        writer.add_scalar('LearningRate/generator', optimizer_g.param_groups[0]['lr'], epoch)
        writer.add_scalar('LearningRate/discriminator', optimizer_d.param_groups[0]['lr'], epoch)

        # Actualizar schedulers
        scheduler_g.step(val_loss)
        scheduler_d.step(train_loss_d)

        # Guardar mejor modelo
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            epochs_no_improve = 0
            torch.save(ema_model.state_dict(), 'unet2D_superres_best.pt')
            print(f"Mejor modelo guardado con loss: {best_val_loss:.6f}")
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience_earlystop:
                print(f"Early stopping en epoch {epoch+1}")
                break
        
        torch.save(ema_model.state_dict(), 'unet2D_superres.pt')

    writer.close()
    print("Entrenamiento completado")
    print(f"Mejor loss alcanzado: {best_val_loss:.6f}")

In [ ]:
train()